#Imports

In [1]:
!pip install woodelf_explainer==0.2.14

In [2]:
import xgboost as xgb
import pandas as pd
import numpy as np
from typing import Union, Dict, Optional, Tuple, Set, List
from math import factorial
import time
from copy import copy
from tqdm import tqdm
from collections import defaultdict
from sklearn.metrics import accuracy_score, f1_score
import scipy
import shap

import lightgbm as lgb

# For GPU execution
# import cupy as cp

In [3]:
from woodelf.parse_models import load_decision_tree_ensemble_model
from woodelf.cube_metric import CubeMetric, ShapleyInteractionValues, ShapleyValues

import woodelf


In [4]:
# Useful if you run this on google colab and downloaded the data into your drive.
# If you run the notebook in other environment remove these lines and change the 'pd.read_csv()' function in this notebook to read from
# where you saved you data
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
# import woodelfHD from Python file in the drive
!cp /content/drive/MyDrive/ShapResearch/HighDepth/AAAI27/woodelfhd.py /content/

import woodelfhd
from woodelfhd import woodelf_for_high_depth

## Environment Note

Use the CPU enviroment with High-RAM option enabled. It is available for subscribed members for Google Colab (toggle the High-RAM option in Runtime->Change runtime type button).

If you only bought computation units, but not a subscription, you can still run the full notebook using the v2-8-TPU machine. This machine have more then enough RAM and it is pretty cheap. Its CPU is a bit slower though, so you will get a slower runtimes.

Final note: preformance in Colab depends on the machine allocated (the CPU option can allocate several types of machines) and on load balancing inside Colab (when more users use the system running times might be slower). So the running times of our algorithms and the shap python package can very. The difference between our approach and the current state-of-the-art is big enough to be noticed, but the excat running times can be slightly different from the ones stated our paper.

# Intrusion Detection Data Loading and Model training

In [6]:
detection_data = pd.read_parquet("drive/MyDrive/ShapResearch/HighDepth/AAAI27/data/KDD_cup_train.parquet")
unlabeled_data = pd.read_parquet("drive/MyDrive/ShapResearch/HighDepth/AAAI27/data/KDD_cup_unlabeled.parquet")
small_test_data = pd.read_parquet("drive/MyDrive/ShapResearch/HighDepth/AAAI27/data/KDD_cup_small_test.parquet")

detection_train_features_names = [f for f in detection_data.columns if f != "target"]
detection_trainset = detection_data[detection_train_features_names]
detection_testset = unlabeled_data[detection_train_features_names]

In [7]:
LIGHTGBM_PARAMS = {
    "boosting_type": "gbdt",
    "objective": "binary",
    "metric": "auc",
    "learning_rate": 0.1,

    # Allow high depth and enough leaves to reach this possible depth
    "num_leaves": 2024,
    "max_depth": 10,
    "min_data_in_leaf": 500,        # Does provide some regulation

    # Sampling (stability + reduces overfit)
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,

    # Practical
    "verbosity": -1,
    "seed": 42,
    "force_col_wise": True,            # often faster/safer for wide data
}


def print_model_stat(model, features):
    mobj = load_decision_tree_ensemble_model(model, features)
    depths = {}
    for tree in mobj.trees:
        for leaf, path in tree.get_all_leaves_with_paths():
            depth = len(path)
            if depth not in depths:
                depths[depth] = 0
            depths[depth] += 1

    for depth in sorted(list(depths.keys())):
        print(f"\tPaths of depth {depth}: {depths.get(depth, 0)} paths")

def lightgbm_model(X_train, y_train, params, num_rounds=100):
    train_set = lgb.Dataset(X_train, label=y_train, free_raw_data=False)
    return lgb.train(
        params=params,
        train_set=train_set,
        num_boost_round=num_rounds
    )


def different_depth_lightgbm_models(trainset, y, params, num_rounds, depths):
    models = {}
    for depth in depths:
        new_params = params.copy()
        new_params['max_depth'] = depth
        models[depth] = lightgbm_model(trainset, y, new_params, num_rounds=num_rounds)
        print("\n\n")
        print(f"Trained on depth {depth}")
        print_model_stat(models[depth], list(trainset.columns))
    return models

In [ ]:
gbm_100trees = different_depth_lightgbm_models(detection_trainset, detection_data['target'], LIGHTGBM_PARAMS, num_rounds=100, depths=[6,9,12,15,18])




Trained on depth 6
	Paths of depth 1: 2 paths
	Paths of depth 2: 10 paths
	Paths of depth 3: 178 paths
	Paths of depth 4: 318 paths
	Paths of depth 5: 607 paths
	Paths of depth 6: 2266 paths



Trained on depth 9
	Paths of depth 2: 20 paths
	Paths of depth 3: 174 paths
	Paths of depth 4: 332 paths
	Paths of depth 5: 596 paths
	Paths of depth 6: 824 paths
	Paths of depth 7: 1133 paths
	Paths of depth 8: 1335 paths
	Paths of depth 9: 3550 paths



Trained on depth 12
	Paths of depth 2: 23 paths
	Paths of depth 3: 179 paths
	Paths of depth 4: 312 paths
	Paths of depth 5: 607 paths
	Paths of depth 6: 815 paths
	Paths of depth 7: 1102 paths
	Paths of depth 8: 1331 paths
	Paths of depth 9: 1557 paths
	Paths of depth 10: 1748 paths
	Paths of depth 11: 1881 paths
	Paths of depth 12: 4902 paths



Trained on depth 15
	Paths of depth 2: 22 paths
	Paths of depth 3: 179 paths
	Paths of depth 4: 322 paths
	Paths of depth 5: 612 paths
	Paths of depth 6: 806 paths
	Paths of depth 7: 1055 paths
	Pa

In [8]:
gbm_10trees = different_depth_lightgbm_models(detection_trainset, detection_data['target'], LIGHTGBM_PARAMS, num_rounds=10, depths=[6,9,12,15,18, 21])




Trained on depth 6
	Paths of depth 3: 29 paths
	Paths of depth 4: 35 paths
	Paths of depth 5: 49 paths
	Paths of depth 6: 170 paths



Trained on depth 9
	Paths of depth 3: 29 paths
	Paths of depth 4: 35 paths
	Paths of depth 5: 51 paths
	Paths of depth 6: 64 paths
	Paths of depth 7: 92 paths
	Paths of depth 8: 96 paths
	Paths of depth 9: 256 paths



Trained on depth 12
	Paths of depth 3: 29 paths
	Paths of depth 4: 35 paths
	Paths of depth 5: 50 paths
	Paths of depth 6: 66 paths
	Paths of depth 7: 84 paths
	Paths of depth 8: 110 paths
	Paths of depth 9: 107 paths
	Paths of depth 10: 130 paths
	Paths of depth 11: 143 paths
	Paths of depth 12: 418 paths



Trained on depth 15
	Paths of depth 3: 29 paths
	Paths of depth 4: 31 paths
	Paths of depth 5: 54 paths
	Paths of depth 6: 70 paths
	Paths of depth 7: 92 paths
	Paths of depth 8: 104 paths
	Paths of depth 9: 121 paths
	Paths of depth 10: 127 paths
	Paths of depth 11: 143 paths
	Paths of depth 12: 177 paths
	Paths of depth 13: 207 

In [ ]:
gbm_1trees = different_depth_lightgbm_models(detection_trainset, detection_data['target'], LIGHTGBM_PARAMS, num_rounds=1, depths=[6,9,12,15,18, 21])




Trained on depth 6
	Paths of depth 3: 3 paths
	Paths of depth 4: 4 paths
	Paths of depth 5: 7 paths
	Paths of depth 6: 10 paths



Trained on depth 9
	Paths of depth 3: 3 paths
	Paths of depth 4: 4 paths
	Paths of depth 5: 7 paths
	Paths of depth 6: 6 paths
	Paths of depth 7: 4 paths
	Paths of depth 8: 3 paths
	Paths of depth 9: 10 paths



Trained on depth 12
	Paths of depth 3: 3 paths
	Paths of depth 4: 4 paths
	Paths of depth 5: 7 paths
	Paths of depth 6: 6 paths
	Paths of depth 7: 4 paths
	Paths of depth 8: 3 paths
	Paths of depth 9: 4 paths
	Paths of depth 10: 9 paths
	Paths of depth 11: 4 paths
	Paths of depth 12: 4 paths



Trained on depth 15
	Paths of depth 3: 3 paths
	Paths of depth 4: 4 paths
	Paths of depth 5: 7 paths
	Paths of depth 6: 6 paths
	Paths of depth 7: 4 paths
	Paths of depth 8: 3 paths
	Paths of depth 9: 4 paths
	Paths of depth 10: 9 paths
	Paths of depth 11: 4 paths
	Paths of depth 13: 4 paths
	Paths of depth 14: 5 paths
	Paths of depth 15: 6 paths



Traine

In [9]:
def high_depth_woodelf_on_models_dict(
        models, consumer_data: pd.DataFrame, background_data: pd.DataFrame, metric: CubeMetric,global_importance: bool = False, GPU=False
    ):
    running_times = {}
    for key, model in models.items():
        start_time = time.time()
        woodelf_for_high_depth(model, consumer_data, background_data, metric=metric, global_importance=global_importance)
        running_time = time.time() - start_time
        running_times[key] = running_time
        print(f"On Depth {key} Took: {running_time}")
    return running_times

In [10]:
def simple_woodelf_on_models_dict(
        models, consumer_data: pd.DataFrame, background_data: pd.DataFrame, metric: CubeMetric, global_importance: bool = False, GPU=False,
    ):
    running_times = {}
    for key, model in models.items():
        start_time = time.time()
        woodelf.simple_woodelf.calculate_background_metric(model, consumer_data, background_data, metric=metric, global_importance=global_importance)
        running_time = time.time() - start_time
        running_times[key] = running_time
        print(f"On Depth {key} Took: {running_time}")
    return running_times

In [11]:
def shap_on_models_dict(models, consumer_data: pd.DataFrame, background_data: pd.DataFrame):
    running_times = {}
    for key, model in models.items():
        start_time = time.time()
        explainer = shap.TreeExplainer(model, background_data, feature_perturbation='interventional')
        simple_shap_values = explainer.shap_values(consumer_data, check_additivity=False)
        running_time = time.time() - start_time
        running_times[key] = running_time
        print(f"On Depth {key} Took: {running_time}")
    return running_times

# Background SHAP

In [ ]:
woodelf_running_times = high_depth_woodelf_on_models_dict(
    {6: gbm_100trees[6], 9: gbm_100trees[9], 12: gbm_10trees[12], 15: gbm_10trees[15], 18: gbm_10trees[18], 21: gbm_1trees[21]},
    consumer_data=detection_testset, background_data=detection_trainset, metric=ShapleyValues(),
    global_importance = False, GPU=False
)
print("Background SHAP: " + " & ".join([str(woodelf_running_times[d]) for d in sorted(list(woodelf_running_times.keys()))]))

Preprocessing the trees and computing SHAP: 100%|██████████| 100/100 [02:32<00:00,  1.53s/it]


M time: 0.0 sec, s time: 0.48 sec (f prepare time: 0.17479658126831055)
On Depth 6 Took: 152.9578673839569


Preprocessing the trees and computing SHAP: 100%|██████████| 100/100 [08:46<00:00,  5.26s/it]


M time: 0.0 sec, s time: 1.58 sec (f prepare time: 0.5371570587158203)
On Depth 9 Took: 526.3879616260529


Preprocessing the trees and computing SHAP: 100%|██████████| 100/100 [20:00<00:00, 12.00s/it]


...

M time: 1.6 sec, s time: 26.84 sec (f prepare time: 3.173158645629883)
On Depth 15 Took: 2434.798138856888


Preprocessing the trees and computing SHAP: 100%|██████████| 100/100 [1:14:32<00:00, 44.73s/it]


M time: 12.76 sec, s time: 124.98 sec (f prepare time: 8.582929849624634)
On Depth 18 Took: 4486.624617815018


Preprocessing the trees and computing SHAP: 100%|██████████| 10/10 [10:05<00:00, 60.56s/it]

M time: 100.69 sec, s time: 40.09 sec (f prepare time: 1.8583738803863525)
On Depth 21 Took: 706.4288716316223
Background SHAP: 152.9578673839569 & 526.3879616260529 & 1201.0850279331207 & 2434.798138856888 & 4486.624617815018 & 706.4288716316223


In [ ]:
woodelf_running_times = simple_woodelf_on_models_dict(
    {6: gbm_100trees[6], 9: gbm_10trees[9], 12: gbm_1trees[12], 15: gbm_1trees[15]}, # Depth 18 crashes due to RAM
    consumer_data=detection_testset, background_data=detection_trainset, metric=ShapleyValues(),
    global_importance = False, GPU=False
)
print("Background SHAP: " + " & ".join([str(woodelf_running_times[d]) for d in sorted(list(woodelf_running_times.keys()))]))

Preprocessing the trees: 100%|██████████| 100/100 [00:57<00:00,  1.74it/s]


cache misses: 74, cache used: 3307, M computation time: 0.16 sec, s computation time: 0.13 sec


Computing the values: 100%|██████████| 100/100 [01:41<00:00,  1.02s/it]


On Depth 6 Took: 159.30461597442627


Preprocessing the trees: 100%|██████████| 100/100 [04:23<00:00,  2.64s/it]


...

On Depth 12 Took: 798.2497398853302


Preprocessing the trees: 100%|██████████| 1/1 [07:21<00:00, 441.54s/it]


cache misses: 24, cache used: 35, M computation time: 431.98 sec, s computation time: 1.36 sec


Computing the values: 100%|██████████| 1/1 [00:03<00:00,  3.14s/it]

On Depth 15 Took: 444.73984932899475
Background SHAP: 159.30461597442627 & 578.1194403171539 & 798.2497398853302 & 444.73984932899475


In [ ]:
train_sample = detection_trainset.sample(10, random_state=42)

shap_running_times = shap_on_models_dict(
    gbm_10trees, consumer_data=detection_testset, background_data=train_sample,
)
print("Background SHAP: " + " & ".join([str(shap_running_times[d]) for d in sorted(list(shap_running_times.keys()))]))

100%|===================| 2980487/2984154 [01:17<00:00]       

On Depth 6 Took: 80.57570219039917


 99%|===================| 2968281/2984154 [02:00<00:00]       

On Depth 9 Took: 123.83157062530518


100%|===================| 2977542/2984154 [03:04<00:00]       

...

On Depth 15 Took: 253.52827334403992


100%|===================| 2980116/2984154 [05:25<00:00]       

On Depth 18 Took: 328.4464371204376


100%|===================| 2981168/2984154 [06:27<00:00]       

On Depth 21 Took: 390.92033886909485
Background SHAP: 80.57570219039917 & 123.83157062530518 & 187.74305200576782 & 253.52827334403992 & 328.4464371204376 & 390.92033886909485


Results on 100 trees...
100%|===================| 2982726/2984154 [08:17<00:00]       On Depth 6 Took: 498.3566279411316
100%|===================| 2982942/2984154 [17:50<00:00]       On Depth 9 Took: 1072.098738193512
100%|===================| 2983857/2984154 [29:39<00:00]       On Depth 12 Took: 1780.6900379657745
100%|===================| 2983347/2984154 [42:18<00:00]       On Depth 15 Took: 2540.6402337551117
 41%|========            | 1220019/2984154 [22:03<31:53]       


# Background SHAP IV

In [ ]:
woodelf_running_times = high_depth_woodelf_on_models_dict(
    {6: gbm_100trees[6], 9: gbm_100trees[9], 12: gbm_10trees[12], 15: gbm_10trees[15], 18: gbm_10trees[18]}, # depth 21 crashes due to RAM (see the crashes on the fraud data)
    consumer_data=detection_testset.sample(10_000, random_state=42), background_data=detection_trainset, metric=ShapleyInteractionValues(),
    global_importance = False, GPU=False
)
print("Background SHAP IV: " + " & ".join([str(woodelf_running_times[d]) for d in sorted(list(woodelf_running_times.keys()))]))

Preprocessing the trees and computing SHAP: 100%|██████████| 100/100 [01:39<00:00,  1.00it/s]


M time: 0.0 sec, s time: 0.67 sec (f prepare time: 0.20899224281311035)
On Depth 6 Took: 99.82319760322571


Preprocessing the trees and computing SHAP: 100%|██████████| 100/100 [05:00<00:00,  3.01s/it]


M time: 0.03 sec, s time: 2.74 sec (f prepare time: 0.6372659206390381)
On Depth 9 Took: 300.93610310554504


Preprocessing the trees and computing SHAP: 100%|██████████| 100/100 [09:41<00:00,  5.81s/it]


M time: 0.5 sec, s time: 15.92 sec (f prepare time: 1.573195219039917)
On Depth 12 Took: 582.1475377082825


Preprocessing the trees and computing SHAP: 100%|██████████| 100/100 [20:22<00:00, 12.22s/it]


M time: 5.6 sec, s time: 192.55 sec (f prepare time: 3.995506525039673)
On Depth 15 Took: 1228.9251835346222


Preprocessing the trees and computing SHAP: 100%|██████████| 10/10 [04:44<00:00, 28.42s/it]

M time: 68.87 sec, s time: 101.83 sec (f prepare time: 0.8862574100494385)
On Depth 18 Took: 353.1802589893341
Background SHAP IV: 99.82319760322571 & 300.93610310554504 & 582.1475377082825 & 1228.9251835346222 & 353.1802589893341


In [ ]:
woodelf_running_times = simple_woodelf_on_models_dict(
    {6: gbm_100trees[6], 9: gbm_100trees[9], 12: gbm_1trees[12]}, # depth 15 crashes due to RAM (see the crashes on the fraud data)
    consumer_data=detection_testset.sample(10_000, random_state=42), background_data=detection_trainset, metric=ShapleyInteractionValues(),
    global_importance = False, GPU=False
)
print("Background SHAP IV: " + " & ".join([str(woodelf_running_times[d]) for d in sorted(list(woodelf_running_times.keys()))]))

Preprocessing the trees: 100%|██████████| 100/100 [01:31<00:00,  1.10it/s]


cache misses: 74, cache used: 3307, M computation time: 0.29 sec, s computation time: 0.33 sec


Computing the values: 100%|██████████| 100/100 [00:02<00:00, 48.36it/s]


On Depth 6 Took: 93.26895594596863


Preprocessing the trees: 100%|██████████| 100/100 [07:00<00:00,  4.21s/it]


...

On Depth 9 Took: 434.9083905220032


Preprocessing the trees: 100%|██████████| 10/10 [29:08<00:00, 174.88s/it]


cache misses: 404, cache used: 768, M computation time: 1685.3 sec, s computation time: 5.78 sec


Computing the values: 100%|██████████| 10/10 [00:11<00:00,  1.19s/it]

On Depth 12 Took: 1760.8616261482239
Background SHAP IV: 93.26895594596863 & 434.9083905220032 & 1760.8616261482239


In [ ]:
# shap Package does not support Background SHAP IV

# Path Dependent SHAP

In [12]:
def pd_high_depth_woodelf_on_models_dict(
        models, consumer_data: pd.DataFrame, metric: CubeMetric, global_importance: bool = False, GPU=False
    ):
    running_times = {}
    for key, model in models.items():
        start_time = time.time()
        woodelf_for_high_depth(model, consumer_data, background_data=None, metric=metric, global_importance=global_importance)
        running_time = time.time() - start_time
        running_times[key] = running_time
        print(f"On Depth {key} Took: {running_time}")
    return running_times

def pd_simple_woodelf_on_models_dict(
        models, consumer_data: pd.DataFrame, metric: CubeMetric, global_importance: bool = False, GPU=False,
    ):
    running_times = {}
    for key, model in models.items():
        start_time = time.time()
        woodelf.simple_woodelf.calculate_path_dependent_metric(model, consumer_data, metric=metric, global_importance=global_importance)
        running_time = time.time() - start_time
        running_times[key] = running_time
        print(f"On Depth {key} Took: {running_time}")
    return running_times

def pd_shap_on_models_dict(models, consumer_data: pd.DataFrame, interaction_values: bool = False):
    running_times = {}
    for key, model in models.items():
        start_time = time.time()
        explainer = shap.TreeExplainer(model, feature_perturbation='tree_path_dependent')
        if interaction_values:
            simple_shap_values = explainer.shap_interaction_values(consumer_data)
        else:
            simple_shap_values = explainer.shap_values(consumer_data, check_additivity=False)
        running_time = time.time() - start_time
        running_times[key] = running_time
        print(f"On Depth {key} Took: {running_time}")
    return running_times

In [ ]:
woodelf_running_times = pd_high_depth_woodelf_on_models_dict(
    {6: gbm_100trees[6], 9: gbm_10trees[9], 12: gbm_10trees[12], 15: gbm_10trees[15], 18: gbm_10trees[18], 21: gbm_1trees[21]},
    consumer_data=detection_testset, metric=ShapleyValues(), global_importance = False, GPU=False
)
print("Path-Dependent SHAP: " + " & ".join([str(woodelf_running_times[d]) for d in sorted(list(woodelf_running_times.keys()))]))

Preprocessing the trees and computing SHAP: 100%|██████████| 100/100 [02:58<00:00,  1.78s/it]


M time: 0.0 sec, s time: 0.61 sec (f prepare time: 0.18261384963989258)
On Depth 6 Took: 178.2783908843994


Preprocessing the trees and computing SHAP: 100%|██████████| 100/100 [10:07<00:00,  6.08s/it]


M time: 0.01 sec, s time: 1.85 sec (f prepare time: 0.5570111274719238)
On Depth 9 Took: 608.1511976718903


Preprocessing the trees and computing SHAP: 100%|██████████| 10/10 [01:51<00:00, 11.19s/it]


...

M time: 1.36 sec, s time: 1.41 sec (f prepare time: 0.26432275772094727)
On Depth 15 Took: 241.63729977607727


Preprocessing the trees and computing SHAP: 100%|██████████| 10/10 [07:24<00:00, 44.44s/it]


M time: 12.92 sec, s time: 6.87 sec (f prepare time: 0.695293664932251)
On Depth 18 Took: 457.4898750782013


Preprocessing the trees and computing SHAP: 100%|██████████| 10/10 [11:21<00:00, 68.13s/it]

M time: 113.98 sec, s time: 31.9 sec (f prepare time: 2.279738426208496)
On Depth 21 Took: 795.5090825557709
Path-Dependent SHAP: 178.2783908843994 & 608.1511976718903 & 112.05137228965759 & 241.63729977607727 & 457.4898750782013 & 795.5090825557709


In [ ]:
woodelf_running_times = pd_simple_woodelf_on_models_dict(
    {6: gbm_100trees[6], 9: gbm_10trees[9], 12: gbm_1trees[12], 15: gbm_1trees[15]}, # Depth 18 crashes due to RAM (see the crashes on the fraud data)
    consumer_data=detection_testset, metric=ShapleyValues(), global_importance = False, GPU=False
)
print("Path-Dependent SHAP: " + " & ".join([str(woodelf_running_times[d]) for d in sorted(list(woodelf_running_times.keys()))]))

Preprocessing the trees: 100%|██████████| 100/100 [00:00<00:00, 247.41it/s]


cache misses: 74, cache used: 3307


Computing the values: 100%|██████████| 100/100 [02:39<00:00,  1.59s/it]


On Depth 6 Took: 159.99326300621033


Preprocessing the trees: 100%|██████████| 100/100 [01:31<00:00,  1.09it/s]


...

On Depth 12 Took: 931.3645303249359


Preprocessing the trees: 100%|██████████| 1/1 [10:11<00:00, 611.65s/it]


cache misses: 24, cache used: 35


Computing the values: 100%|██████████| 1/1 [00:05<00:00,  5.15s/it]

On Depth 15 Took: 616.8218574523926
Path-Dependent SHAP: 159.99326300621033 & 593.4511578083038 & 931.3645303249359 & 616.8218574523926


In [ ]:
shap_running_times = pd_shap_on_models_dict(
    {6: gbm_10trees[6], 9: gbm_10trees[9], 12: gbm_10trees[12], 15: gbm_10trees[15], 18: gbm_1trees[18], 21: gbm_1trees[21]}, consumer_data=detection_testset,
)
print("Path-Dependent SHAP: " + " & ".join([str(shap_running_times[d]) for d in sorted(list(shap_running_times.keys()))]))

For 100 trees:


/usr/local/lib/python3.12/dist-packages/shap/explainers/_tree.py:620: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(
On Depth 6 Took: 509.0996820926666
/usr/local/lib/python3.12/dist-packages/shap/explainers/_tree.py:620: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(
On Depth 9 Took: 1862.2444863319397
/usr/local/lib/python3.12/dist-packages/shap/explainers/_tree.py:620: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(
On Depth 12 Took: 4518.586185932159


# Path Dependent SHAP IV

In [ ]:
woodelf_running_times = pd_high_depth_woodelf_on_models_dict(
    {6: gbm_100trees[6], 9: gbm_100trees[9], 12: gbm_10trees[12], 15: gbm_10trees[15], 18: gbm_10trees[18]}, # depth 21 crashes due to RAM (see the crashes on the fraud data)
    consumer_data=detection_testset.sample(10_000, random_state=42), metric=ShapleyInteractionValues(), global_importance = False, GPU=False
)
print("Path-Dependent SHAP IV: " + " & ".join([str(woodelf_running_times[d]) for d in sorted(list(woodelf_running_times.keys()))]))

Preprocessing the trees and computing SHAP: 100%|██████████| 100/100 [00:02<00:00, 33.66it/s]


M time: 0.0 sec, s time: 0.5 sec (f prepare time: 0.14903044700622559)
On Depth 6 Took: 4.08766770362854


Preprocessing the trees and computing SHAP: 100%|██████████| 100/100 [00:12<00:00,  7.74it/s]


M time: 0.02 sec, s time: 2.24 sec (f prepare time: 0.5481154918670654)
On Depth 9 Took: 13.229509592056274


Preprocessing the trees and computing SHAP: 100%|██████████| 10/10 [00:03<00:00,  3.31it/s]


M time: 0.52 sec, s time: 0.75 sec (f prepare time: 0.10486459732055664)
On Depth 12 Took: 3.5779616832733154


Preprocessing the trees and computing SHAP: 100%|██████████| 10/10 [00:10<00:00,  1.01s/it]


M time: 5.19 sec, s time: 5.04 sec (f prepare time: 0.23962712287902832)
On Depth 15 Took: 15.352846384048462


Preprocessing the trees and computing SHAP: 100%|██████████| 10/10 [00:58<00:00,  5.82s/it]

M time: 69.39 sec, s time: 44.46 sec (f prepare time: 0.6762969493865967)
On Depth 18 Took: 127.7246720790863
Path-Dependent SHAP IV: 4.08766770362854 & 13.229509592056274 & 3.5779616832733154 & 15.352846384048462 & 127.7246720790863


In [ ]:
woodelf_running_times = pd_simple_woodelf_on_models_dict(
    {6: gbm_100trees[6], 9: gbm_100trees[9], 12: gbm_1trees[12]}, # depth 15 crashes due to RAM (see the crashes on the fraud data)
    consumer_data=detection_testset.sample(10_000, random_state=42), metric=ShapleyInteractionValues(), global_importance = False, GPU=False
)
print("Path-Dependent SHAP IV: " + " & ".join([str(woodelf_running_times[d]) for d in sorted(list(woodelf_running_times.keys()))]))

Preprocessing the trees: 100%|██████████| 100/100 [00:00<00:00, 150.27it/s]


cache misses: 74, cache used: 3307


Computing the values: 100%|██████████| 100/100 [00:01<00:00, 57.15it/s]


On Depth 6 Took: 2.546661376953125


Preprocessing the trees: 100%|██████████| 100/100 [02:19<00:00,  1.39s/it]


...

On Depth 9 Took: 150.1194577217102


Preprocessing the trees: 100%|██████████| 10/10 [28:50<00:00, 173.07s/it]


cache misses: 404, cache used: 768


Computing the values: 100%|██████████| 10/10 [00:07<00:00,  1.36it/s]

On Depth 12 Took: 1738.2089891433716
Path-Dependent SHAP IV: 2.546661376953125 & 150.1194577217102 & 1738.2089891433716


In [13]:
shap_running_times = pd_shap_on_models_dict(
    gbm_10trees, consumer_data=detection_testset.sample(10_000, random_state=42), interaction_values=True
)
print("Path-Dependent SHAP IV: " + " & ".join([str(shap_running_times[d]) for d in sorted(list(shap_running_times.keys()))]))

On Depth 6 Took: 15.71250605583191
On Depth 9 Took: 76.22914457321167
On Depth 12 Took: 226.68086862564087
On Depth 15 Took: 521.770471572876
On Depth 18 Took: 986.493353843689
On Depth 21 Took: 1677.5858938694
Path-Dependent SHAP IV: 15.71250605583191 & 76.22914457321167 & 226.68086862564087 & 521.770471572876 & 986.493353843689 & 1677.5858938694


On 100 trees

On Depth 6 Took: 213.7403702735901
On Depth 9 Took: 1082.4352631568909
On Depth 12 Took: 3268.0648460388184

